In [1]:
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset,Dataset
from huggingface_hub import login,HfApi
import torch
import json

# Đăng nhập vào huggingface account
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("translating")
login(secret_value_0)


# Truy cập vào dataset
doctorSet = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")

# Xác định device (GPU nếu có, ngược lại CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng device: {device}")

# Lấy model dịch thuật và token đi kèm với nó ra
model_name = "VietAI/envit5-translation"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# instruction là cố định
instruction = "Nếu bạn là bác sĩ, vui lòng trả lời các câu hỏi y khoa dựa trên mô tả của bệnh nhân."
#print(f"Chiều dài của tập dữ liệu: {len(doctorSet['train'])}")

# Lưu dữ liệu vào file JSON trong thư mục output
output_dir = "/kaggle/working/output_data.jsonl"  # Đường dẫn nơi lưu file
with open(output_dir, "w", encoding="utf-8") as f:
    for i in range(2000,3500): # Tuỳ chọn trong việc lấy mẫu của dataset
        inputs = doctorSet["train"][i]["input"]
        output = doctorSet["train"][i]["output"]
        # Xử lý đơn lẻ từng thành phần text cần dịch bằng GPU song song giúp tác vụ nhanh hơn
        input_token = tokenizer([inputs],return_tensors = "pt",padding = True).to(device)
        translated_input = model.generate(**input_token,max_length = 700,num_beams = 4)
        translated_text_input = tokenizer.batch_decode(translated_input.cpu(), skip_special_tokens=True)
        output_token = tokenizer([output],return_tensors = "pt",padding = True).to(device)
        translated_output = model.generate(**output_token,max_length = 700,num_beams = 4)
        translated_text_output = tokenizer.batch_decode(translated_output.cpu(), skip_special_tokens=True)
        # Đưa các thông tin vào file kết quả
        data = {
            "instruction": instruction,
            "input": translated_text_input[0],
            "output": translated_text_output[0]
        }
        # Ghi vào kết quả trong file output
        f.write(json.dumps(data,ensure_ascii=False) + "\n")
        # Giải phóng bộ nhớ sau mỗi lần lặp
        del input_token, output_token, translated_input, translated_output, translated_text_input,translated_text_output
        torch.cuda.empty_cache()
        # In tiến trình xử lý
        if (i + 1) % 10 == 0:
            print(f"Đã xử lý {i + 1} mẫu")


# Đẩy lên huggingface
# ! mỗi lần up cần thay direct mới để tránh ghi đè
api = HfApi(token=secret_value_0)
api.upload_file(
    path_or_fileobj=output_dir,  # Đường dẫn tới file
    path_in_repo="output_data.json",  # Tên file trong Hugging Face
    repo_id="PhongGoldFish/chatDoctor",  # Repo ID của bạn trên Hugging Face
    repo_type="dataset",  # Loại repo (dataset)
)

#dataset_final.push_to_hub("PhongGoldFish/chatDoctor")
print("200 Done")

Sử dụng device: cuda


2025-05-12 14:38:24.644780: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747060704.668739     389 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747060704.675873     389 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Đã xử lý 2010 mẫu
Đã xử lý 2020 mẫu
Đã xử lý 2030 mẫu
Đã xử lý 2040 mẫu
Đã xử lý 2050 mẫu
Đã xử lý 2060 mẫu
Đã xử lý 2070 mẫu
Đã xử lý 2080 mẫu
Đã xử lý 2090 mẫu
Đã xử lý 2100 mẫu
Đã xử lý 2110 mẫu
Đã xử lý 2120 mẫu
Đã xử lý 2130 mẫu
Đã xử lý 2140 mẫu
Đã xử lý 2150 mẫu
Đã xử lý 2160 mẫu
Đã xử lý 2170 mẫu
Đã xử lý 2180 mẫu
Đã xử lý 2190 mẫu
Đã xử lý 2200 mẫu
Đã xử lý 2210 mẫu
Đã xử lý 2220 mẫu
Đã xử lý 2230 mẫu
Đã xử lý 2240 mẫu
Đã xử lý 2250 mẫu
Đã xử lý 2260 mẫu
Đã xử lý 2270 mẫu
Đã xử lý 2280 mẫu
Đã xử lý 2290 mẫu
Đã xử lý 2300 mẫu
Đã xử lý 2310 mẫu
Đã xử lý 2320 mẫu
Đã xử lý 2330 mẫu
Đã xử lý 2340 mẫu
Đã xử lý 2350 mẫu
Đã xử lý 2360 mẫu
Đã xử lý 2370 mẫu
Đã xử lý 2380 mẫu
Đã xử lý 2390 mẫu
Đã xử lý 2400 mẫu
Đã xử lý 2410 mẫu
Đã xử lý 2420 mẫu
Đã xử lý 2430 mẫu
Đã xử lý 2440 mẫu
Đã xử lý 2450 mẫu
Đã xử lý 2460 mẫu
Đã xử lý 2470 mẫu
Đã xử lý 2480 mẫu
Đã xử lý 2490 mẫu
Đã xử lý 2500 mẫu
Đã xử lý 2510 mẫu
Đã xử lý 2520 mẫu
Đã xử lý 2530 mẫu
Đã xử lý 2540 mẫu
Đã xử lý 2550 mẫu
Đã xử lý 2

In [ ]:
%pip install -U transformers
%pip install -U datasets
%pip install huggingface_hub
%pip install torch

In [ ]:
print(translated_text_input[0])
print(translated_text_output[0])

In [ ]:
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset,Dataset
from huggingface_hub import login,HfApi
import torch
import json
print("200 Ok")

In [ ]:
import os

file_path = "/kaggle/working/output_data.jsonl"

if os.path.exists(file_path):
    os.remove(file_path)
    print(f"Đã xoá file: {file_path}")
else:
    print("File không tồn tại.")